# Summarization & Entity Recognition

This notebook builds structured product insights from reviews and metadata, then exports a unified profile file for downstream semantic search and demo notebooks.

It supports the objective of extracting actionable product intelligence from customer feedback using entity extraction and concise summaries.


## Setup

Load dependencies, configuration paths, and helper utilities required for entity extraction and summarization.


In [2]:
import os
import sys
sys.path.insert(0, "src")

import warnings
import pandas as pd
import spacy

from mean_squared_terrors.config import IO_DIR
from mean_squared_terrors.summarization import (
    get_device,
    build_entities_dataframe,
    build_summarizer,
    run_batch_summarization,
    build_product_profiles,
)

warnings.filterwarnings("ignore")

DATA_DIR = IO_DIR
DEVICE = get_device()
print(f"Device for transformers: {'GPU' if DEVICE == 0 else 'CPU'}")

Products: 7,647  |  Reviews: 318,258
Device for transformers: CPU


## Load Data

In [ ]:
products = pd.read_csv(os.path.join(DATA_DIR, "products_cleaned.csv"))
reviews  = pd.read_csv(os.path.join(DATA_DIR, "reviews_cleaned.csv"))

print(f"Products: {len(products):,}  |  Reviews: {len(reviews):,}")

## Entity Recognition

Extract structured attributes from product text and reviews:

1. Ingredients (regex-based patterns)
2. Product concerns and use-cases
3. Suitable skin types


In [20]:
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

entities_df = build_entities_dataframe(products, reviews, nlp)
print(f"\nEntities extracted for {len(entities_df):,} products")
print(f"With at least 1 ingredient:     {(entities_df['n_ingredients']>0).mean():.1%}")
print(f"With at least 1 certification:  {(entities_df['n_certifications']>0).mean():.1%}")
print(f"With at least 1 use case:        {(entities_df['use_cases']!='').mean():.1%}")

entities_df.to_csv(os.path.join(DATA_DIR, "product_entities.csv"), index=False)
print("\nSaved product_entities.csv")

print("\n--- 3 examples ---")
for _, r in entities_df[entities_df['n_ingredients'] > 2].head(3).iterrows():
    print(f"\n{r['product_title'][:90]}")
    print(f"  ingredients: {r['ingredients']}")
    print(f"  certifications: {r['certifications']}")
    print(f"  use_cases: {r['use_cases']}")

Loading spaCy model...
Extracting entities from 7,647 products...
  1,000/7,647
  2,000/7,647
  3,000/7,647
  4,000/7,647
  5,000/7,647
  6,000/7,647
  7,000/7,647

Entities extracted for 7,647 products
With at least 1 ingredient:     8.8%
With at least 1 certification:  19.6%
With at least 1 use case:        13.1%

Saved product_entities.csv

--- 3 examples ---

Avon SKIN-SO-SOFT Bug Guard PLUS IR3535 Insect Repellent Moisturizing Lotion - SPF 30 Gent
  ingredients: vitamin e | spf30 | spf 30
  certifications: 
  use_cases: 

Kleenex Expressions Soothing Lotion Facial Tissues, 18 Cube Boxes, 65 Tissues Per Box 1,17
  ingredients: vitamin e | coconut oil | aloe vera
  certifications: natural
  use_cases: 

Plant Based Protein Powder - Kids Protein Shake with Multivitamins - Kids Vitamin and Prob
  ingredients: probiotics | vitamin d3 | vitamin c
  certifications: 
  use_cases: 


## Review Summarization

Generate concise positive and negative summaries per product with a two-stage process:

1. Extract informative sentences from review buckets
2. Produce final summaries with the summarization model


In [3]:
summarizer = build_summarizer(DEVICE)

Loading BART summarization model...
(first run: downloads ~1.6GB — cached afterwards)


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model loaded in 2.4s


In [23]:
print("Starting batch summarization...")
print("Estimated time: long run depending on hardware and model cache.")
print("Press Interrupt to stop — partial results are saved every 500 products.\n")

summaries_df = run_batch_summarization(
    products=products,
    reviews=reviews,
    summarizer=summarizer,
    data_dir=DATA_DIR,
    save_every=500,
)
summaries_df.to_csv(os.path.join(DATA_DIR, "product_summaries.csv"), index=False)

print(f"Products with BART summary:    {(summaries_df['method']=='model').sum():,}")
print(f"Products with extractive:      {(summaries_df['method']=='extractive').sum():,}")
print("Saved product_summaries.csv")


Starting batch summarization...
Estimated time: long run depending on hardware and model cache.
Press Interrupt to stop — partial results are saved every 500 products.

Starting batch summarization...
  100/7,647  |  1.4 min elapsed  |  ~106 min remaining
  200/7,647  |  2.7 min elapsed  |  ~102 min remaining
  300/7,647  |  4.1 min elapsed  |  ~99 min remaining
  400/7,647  |  5.4 min elapsed  |  ~98 min remaining
  500/7,647  |  6.7 min elapsed  |  ~96 min remaining
  [checkpoint saved: 500 products]
  600/7,647  |  8.0 min elapsed  |  ~94 min remaining
  700/7,647  |  9.3 min elapsed  |  ~93 min remaining
  800/7,647  |  10.6 min elapsed  |  ~91 min remaining
  900/7,647  |  11.9 min elapsed  |  ~89 min remaining
  1,000/7,647  |  13.1 min elapsed  |  ~87 min remaining
  [checkpoint saved: 1000 products]
  1,100/7,647  |  14.4 min elapsed  |  ~86 min remaining
  1,200/7,647  |  15.7 min elapsed  |  ~84 min remaining
  1,300/7,647  |  16.9 min elapsed  |  ~83 min remaining
  1,400/7,

## Quality Check

Validate output coverage and inspect summary/entity quality before exporting final artifacts.


In [24]:
if 'summaries_df' not in dir() or len(summaries_df) == 0:
    summaries_df = pd.read_csv(os.path.join(DATA_DIR, "product_summaries.csv"))
    print(f"Reloaded: {len(summaries_df):,} summaries")

print(f"Total summaries: {len(summaries_df):,}")
print(f"With non-empty summary: {(summaries_df['summary_full']!='').mean():.1%}")
print(f"With pros: {(summaries_df['pros']!='').mean():.1%}")
print(f"With cons: {(summaries_df['cons']!='').mean():.1%}")
print(f"BART method: {(summaries_df['method']=='model').mean():.1%}")

print("\n--- Sample summaries (BART method) ---")
model_summs = summaries_df[
    (summaries_df['method'] == 'model')
    & (summaries_df['pros'] != '')
    & (summaries_df['cons'] != '')
].head(5)

for _, r in model_summs.iterrows():
    prod_title = products[products['parent_asin'] == r['parent_asin']]['product_title'].values
    title = prod_title[0][:60] if len(prod_title) > 0 else r['parent_asin']
    print(f"\n{title}")
    print(f"  Summary: {r['summary_full']}")
    print(f"  Pros:    {r['pros'][:120]}")
    print(f"  Cons:    {r['cons'][:120]}")
    if r['best_for']:
        print(f"  Best for: {r['best_for'][:100]}")


Total summaries: 7,647
With non-empty summary: 100.0%
With pros: 97.1%
With cons: 75.1%
BART method: 100.0%

--- Sample summaries (BART method) ---

High Potency Magnesium Citrate Capsules 1000mg - Gentle Comp
  Summary: Product: High Potency Magnesium Citrate Capsules 1000mg - Gentle Complex for Max Absorption, Made in USA, Bes. Negative reviews: After taking this supplement for several days I found my constipation issue were much worse.
  Pros:    This review is more to clarify someone else's review bc they didn't understand understand the labeling!; It shows 1000mg
  Cons:    After taking this supplement for several days I found my constipation issue were much worse and I was bloating a lot.

Dr. Foot's Gel Heel Protectors, Plantar Fasciitis Treatment 
  Summary: Product: Dr. Foot's Gel Heel Protectors, Plantar Fasciitis Treatment - Breathable Silicone Heel Cushion Cups.
  Pros:    Easy to put on, comfortable, fits in shoe easily, adds lots of extra heel cushion.; For the most part 

## Merge with Embedding Index

Combine summaries and extracted entities into `product_profiles.csv`, the profile table used by semantic search and demo notebooks.


In [25]:
idx = build_product_profiles(DATA_DIR, summaries_df)
idx.to_csv(os.path.join(DATA_DIR, "product_profiles.csv"), index=False)

print(f"product_profiles.csv saved: {len(idx):,} rows")
print(f"Columns: {list(idx.columns)}")
print("\nCoverage:")
print(f"  With summary:      {idx['summary_full'].notna().mean():.1%}")
print(f"  With pros:         {(idx['pros'].fillna('')!='').mean():.1%}")
print(f"  With ingredients:  {(idx['ingredients'].fillna('')!='').mean():.1%}")
print(f"  With certifications: {(idx['certifications'].fillna('')!='').mean():.1%}")

product_profiles.csv saved: 7,604 rows
Columns: ['parent_asin', 'product_title', 'brand', 'product_avg_rating', 'product_rating_count', 'price', 'price_bucket', 'alpha', 'emb_row', 'n_reviews', 'pct_positive', 'pct_negative', 'total_helpful', 'avg_rating_reviews', 'summary_full', 'pros', 'cons', 'best_for', 'method', 'ingredients', 'certifications', 'use_cases', 'n_ingredients', 'n_certifications']

Coverage:
  With summary:      99.9%
  With pros:         97.2%
  With ingredients:  8.7%
  With certifications: 19.6%


## Output Coverage Summary

Per-product coverage of the structured artefacts produced by this notebook, measured against the production catalogue (the 7,593 products in the FAISS index, joined with `product_summaries.csv` and `product_entities.csv`). Full breakdown — including the top 15 ingredients and certifications extracted — lives in `evaluation/output/summarization_stats.json` and is summarised in `evaluation/08_Evaluation.ipynb` §8.


In [ ]:
import json, os

STATS = '../evaluation/output/summarization_stats.json'
if not os.path.isfile(STATS):
    STATS = 'evaluation/output/summarization_stats.json'

with open(STATS) as f:
    stats = json.load(f)

rows = [
    ('summary_full',     stats['summary_coverage_pct']),
    ('pros',              stats['pros_coverage_pct']),
    ('cons',              stats['cons_coverage_pct']),
    ('ingredients (≥1)',  stats['ingredients_coverage_pct']),
    ('certifications (≥1)', stats['cert_coverage_pct']),
]
import pandas as pd
df = pd.DataFrame(rows, columns=['field', 'coverage_pct'])
df['coverage_pct'] = df['coverage_pct'].round(1)
df


### Summarisation method split

Products are summarised with BART-Large-CNN when ≥5 informative reviews exist; otherwise we fall back to **extractive** summarisation (top sentences) to avoid hallucination on thin evidence.


In [ ]:
method_dist = stats['method_dist']
model_pct = method_dist.get('model', 0) / sum(method_dist.values()) * 100
extr_pct  = method_dist.get('extractive', 0) / sum(method_dist.values()) * 100
print(f"BART-generated   : {method_dist.get('model', 0):>5,} ({model_pct:.1f}%)")
print(f"Extractive (fallback): {method_dist.get('extractive', 0):>5,} ({extr_pct:.1f}%)")
print(f"Avg summary length   : {stats['summary_avg_chars']:.0f} chars")
print(f"Avg entities/product  : {stats['avg_ingredients']:.2f} ingredients, {stats['avg_certs']:.2f} certs")


### Top extracted entities

Most frequent ingredients and certifications across the catalogue. Useful both as sanity-check (the top entries are real domain terms, not noise) and as material for the demo's faceted-filter UI.


In [ ]:
top_ings  = pd.Series(stats['top_ingredients']).sort_values(ascending=False)
top_certs = pd.Series(stats['top_certifications']).sort_values(ascending=False)
import pandas as pd
tbl = pd.concat([top_ings.head(10).rename('count'), top_certs.head(10).rename('count')],
                keys=['Ingredients', 'Certifications'])
tbl


**Take-away:** the structured-output pipeline (BART summaries + regex/spaCy entity extraction) is the *non-search* asset of the project. It transforms raw review text into the kind of structured product profile that an e-commerce search system can query, filter and rank — going beyond what a vanilla retrieval engine would produce.
